# M3 lexical baseline vs human labels

Compare primary human labels (`human_annotation`) with the deterministic lexical overlap `baseline_scores.m3.lexical.sequence_ratio` from Milestone 3 phase 1.

**Prerequisite:** run from the repository root (or adjust `REPO` below) and use an editable install or `PYTHONPATH=.` so `rde_eval` imports resolve.

Generate `results/samples_with_m3.jsonl` first:

```bash
python scripts/run_baselines.py --input data/samples.jsonl --output results/samples_with_m3.jsonl
```

In [ ]:
from __future__ import annotations

import json
from collections import defaultdict
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").exists() and (ROOT.parent / "pyproject.toml").exists():
    ROOT = ROOT.parent

PATH = ROOT / "results" / "samples_with_m3.jsonl"
if not PATH.is_file():
    PATH = ROOT / "data" / "samples.jsonl"
    print("Using data/samples.jsonl — run run_baselines.py to create results/samples_with_m3.jsonl")

print("ROOT =", ROOT)
print("JSONL =", PATH)

In [ ]:
def load_jsonl(path: Path) -> list[dict]:
    rows: list[dict] = []
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


rows = load_jsonl(PATH)
len(rows)

In [ ]:
from rde_eval.baselines import merge_milestone3_baseline

prepared: list[dict] = []
for r in rows:
    r = dict(r)
    if "baseline_scores" not in r or "m3" not in (r.get("baseline_scores") or {}):
        r["baseline_scores"] = merge_milestone3_baseline(
            r.get("baseline_scores"), source=str(r["source"]), output=str(r["output"])
        )
    prepared.append(r)

rows = prepared

## Mean lexical ratio by human primary label

Exploratory: higher overlap does not imply RDE "Preserved" — this is only a string-similarity proxy.

In [ ]:
by_label: dict[str, list[float]] = defaultdict(list)
for r in rows:
    label = r.get("human_annotation")
    if not label:
        continue
    ratio = r["baseline_scores"]["m3"]["lexical"]["sequence_ratio"]
    by_label[str(label)].append(ratio)

for label in sorted(by_label):
    vals = by_label[label]
    mean = sum(vals) / len(vals)
    print(f"{label:22}  mean_ratio={mean:.4f}  n={len(vals)}")